# Fed-Drift — a quantitative teardown 🔬
### The pre-FOMC drift table · the Welch t · the pre/post-publication decay

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Still an edge after publication?: Not supported](https://img.shields.io/badge/Still_an_edge_after_publication%3F-Not_supported-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). A real, economically enormous calendar effect — and a clean case of publication killing an anomaly.

> ⚠️ **Not investment advice.** SPY daily returns vs 264 scheduled FOMC announcements, 1993–2026 (Yahoo + Fed calendar). Daily close-to-close understates the intraday drift. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (fed_drift/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from fed_drift import data, strategy as st
ret, fomc = data.fetch_panel()                 # cache-first (shared SPY pull)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Real** | 3% of days → 11.5% of return (19.8% pre-2011) |
| Tradability | **Fragile** | pre-FOMC mean +0.24%→+0.09%, t 1.7→0.4 |
| Edge after publication? | **Not supported** | McLean-Pontiff decay |

> 💡 *In plain words:* a real giant, arbitraged away in plain sight.

## 1 · The claim, steelmanned

- **H₁:** pre-FOMC days have higher mean returns than other days.
- **H₂:** the effect is economically large (carries a big share of total return).
- **H₃:** it persists post-publication (still tradable today).

## 2 · So what? — what rides on each

H₁/H₂ confirm a real, scheduled calendar effect. H₃ decides whether it's a live trade or an economic-history footnote. A decayed anomaly is the single most common fate of a published one.

## 3 · How we'd know — the protocol

Tag the session before each FOMC announcement → Welch t of pre-FOMC vs rest → the share of cumulative return on pre-FOMC days → split at 2011 to test decay. Daily close-to-close is a conservative (noisy) proxy for the intraday window.

## 4 · The teardown

### 4.1 The full-sample drift

In [2]:
t=st.drift_table(ret,fomc,lead=1)
print({k:(round(v,5) if isinstance(v,float) else v) for k,v in t.items()})

{'pre_mean': 0.00174, 'rest_mean': 0.00043, 'pre_ann': 0.01377, 'tstat': 1.58962, 'n_pre': 264, 'n_total': 8398, 'pre_share': 0.11522, 'pre_sum': 0.4588, 'total_sum': 3.98207}


> 💡 *In plain words:* pre-FOMC +0.174%/day vs +0.043%, diff t≈1.6 (noisy daily data), but **3.1% of sessions carry 11.5% of cumulative return**. **H₁ and H₂ hold.**

### 4.2 The pre/post-publication split — H₃ fails

In [3]:
sp=st.split_by_date(ret,fomc,cut='2011-01-01')
display(pd.DataFrame({e:v for e,v in sp.items()}).T[['pre_mean','rest_mean','tstat','pre_share','n_pre']].round(4))

,pre_mean,rest_mean,tstat,pre_share,n_pre
pre_publication,0.0024,0.0003,1.6789,0.1981,141.0
post_publication,0.0009,0.0006,0.3703,0.0518,124.0


Before 2011 the pre-FOMC day averaged +0.243%/day (t≈1.7) and its ~3% of sessions earned **19.8%** of the market's total return. After 2011: +0.094%/day, t≈0.4, just **5.2%** of return. **H₃ rejected** — publication front-ran the drift into insignificance, the textbook McLean-Pontiff (2016) decay.

## 5 · The verdict

H₁/H₂ hold, H₃ rejected → Signal `REAL`, Tradability `FRAGILE`, edge-after-publication `NOT SUPPORTED`. The fingerprinted run is in [`docs/results.md`](../docs/results.md).

## 6 · Could you trade it?

Only with a time machine to before 2011. The drift was real and large, but a publicly-scheduled, widely-published calendar is trivially front-run — which is exactly what the post-2011 collapse shows. Today it's risk-management trivia, not a trade.

## 7 · Going further

Forks: (a) the intraday 2pm-to-2pm window (the true effect); (b) the FOMC even-week cycle (Cieslak-Morse-Vissing-Jørgensen); (c) other scheduled releases (CPI, NFP); (d) the international spillover Lucca-Moench document. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).